# Order Book Analysis & Market Making

Analyzes live order book snapshots + deltas (Apr 18-20, 2026) to understand:
- Bid-ask spread by market series
- Depth at top of book
- Book imbalance as a directional signal
- Market making feasibility

**Key finding:** KXNBAPTS median spread $0.03 (81% of snapshots profitable after fees).
Imbalance signal is inconclusive (corr -0.04, n=112). Only ~30 hours of data — needs
more collection before conclusions are reliable.

In [ ]:
import json
import gzip
import re
import gc
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone, timedelta

import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

s3 = boto3.client("s3")
S3_BUCKET = "prediction-markets-data"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

## 2. Order Book Analysis

Using live order book snapshots + deltas to understand:
- Typical bid-ask spread (can we execute passively?)
- Depth at top of book (how much size can we trade?)
- Book imbalance as a directional signal

In [ ]:
# Load all order book snapshots
paginator = s3.get_paginator("list_objects_v2")
snap_keys = []
for page in paginator.paginate(Bucket=S3_BUCKET, Prefix="bronze/kalshi_ws/orderbook_snapshot/"):
    snap_keys.extend([o["Key"] for o in page.get("Contents", [])])
print(f"Order book snapshot files: {len(snap_keys)}")

# Parse all snapshots
snapshots = []
for key in snap_keys:
    obj = s3.get_object(Bucket=S3_BUCKET, Key=key)
    data = gzip.decompress(obj["Body"].read()).decode()
    for line in data.strip().split("\n"):
        rec = json.loads(line)
        msg = rec["frame"]["msg"]
        
        # Parse the book: yes_dollars_fp is the bid side, no_dollars_fp is the ask
        # Format: [[price, size], ...]
        yes_levels = msg.get("yes_dollars_fp", [])
        no_levels = msg.get("no_dollars_fp", [])
        
        # Best bid = highest yes price with size, best ask = 1 - highest no price with size
        # On Kalshi: yes_bid is what buyers will pay for YES
        # no_bid is what buyers will pay for NO = (1 - ask_for_yes)
        best_yes_bid = max([float(p) for p, s in yes_levels]) if yes_levels else None
        best_no_bid = max([float(p) for p, s in no_levels]) if no_levels else None
        best_yes_ask = 1.0 - best_no_bid if best_no_bid else None
        
        # Total depth
        yes_depth = sum(float(s) for p, s in yes_levels) if yes_levels else 0
        no_depth = sum(float(s) for p, s in no_levels) if no_levels else 0
        
        # Top-of-book depth (at best price)
        top_yes_size = float(yes_levels[-1][1]) if yes_levels else 0  # levels sorted asc
        best_yes_price_level = max(yes_levels, key=lambda x: float(x[0])) if yes_levels else None
        top_bid_size = float(best_yes_price_level[1]) if best_yes_price_level else 0
        
        snapshots.append({
            "t_receipt": rec["t_receipt"],
            "ticker": msg["market_ticker"],
            "best_bid": best_yes_bid,
            "best_ask": best_yes_ask,
            "spread": (best_yes_ask - best_yes_bid) if (best_yes_ask and best_yes_bid) else None,
            "yes_depth": yes_depth,
            "no_depth": no_depth,
            "bid_depth_top": top_bid_size,
            "n_yes_levels": len(yes_levels),
            "n_no_levels": len(no_levels),
        })

snap_df = pd.DataFrame(snapshots)
snap_df["t_receipt_dt"] = pd.to_datetime(snap_df["t_receipt"], unit="s", utc=True)
print(f"\nParsed {len(snap_df)} order book snapshots across {snap_df['ticker'].nunique()} tickers")
print(f"Time range: {snap_df['t_receipt_dt'].min()} to {snap_df['t_receipt_dt'].max()}")

In [ ]:
# Bid-ask spread analysis
valid_spreads = snap_df[snap_df["spread"].notna() & (snap_df["spread"] > 0)]
print("BID-ASK SPREAD ANALYSIS")
print("=" * 60)
print(f"  Valid observations: {len(valid_spreads)}")
print(f"  Mean spread: ${valid_spreads['spread'].mean():.4f}")
print(f"  Median spread: ${valid_spreads['spread'].median():.4f}")
print(f"  p25: ${valid_spreads['spread'].quantile(0.25):.4f}")
print(f"  p75: ${valid_spreads['spread'].quantile(0.75):.4f}")
print(f"  Spread <= $0.01: {(valid_spreads['spread'] <= 0.01).mean():.1%}")
print(f"  Spread <= $0.02: {(valid_spreads['spread'] <= 0.02).mean():.1%}")
print(f"  Spread <= $0.03: {(valid_spreads['spread'] <= 0.03).mean():.1%}")

# By market type (GAME vs SPREAD vs TOTAL vs player props)
valid_spreads = valid_spreads.copy()
valid_spreads["series"] = valid_spreads["ticker"].apply(
    lambda t: re.match(r"(KXNBA\w+?)-", t).group(1) if re.match(r"(KXNBA\w+?)-", t) else "unknown"
)
print(f"\nSpread by market series:")
for series, grp in valid_spreads.groupby("series"):
    print(f"  {series:15s}: median ${grp['spread'].median():.3f}, "
          f"mean ${grp['spread'].mean():.3f}, n={len(grp)}")

# Depth at top of book
print(f"\nTOP-OF-BOOK DEPTH (yes side):")
print(f"  Mean: ${valid_spreads['bid_depth_top'].mean():,.0f}")
print(f"  Median: ${valid_spreads['bid_depth_top'].median():,.0f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(valid_spreads["spread"], bins=50, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axvline(valid_spreads["spread"].median(), color="red", linestyle="--", 
                label=f"median=${valid_spreads['spread'].median():.3f}")
axes[0].set_xlabel("Bid-Ask Spread ($)")
axes[0].set_ylabel("Count")
axes[0].set_title("Bid-Ask Spread Distribution")
axes[0].legend()

# Spread by series
series_medians = valid_spreads.groupby("series")["spread"].median().sort_values()
axes[1].barh(series_medians.index, series_medians.values, color="darkorange")
axes[1].set_xlabel("Median Spread ($)")
axes[1].set_title("Median Spread by Market Series")
plt.tight_layout()
plt.show()

In [ ]:
# Order book imbalance as directional signal
# Imbalance = (yes_depth - no_depth) / (yes_depth + no_depth)
# Hypothesis: when yes side is thicker, price is more likely to move up (more buyers waiting)

snap_df["imbalance"] = (snap_df["yes_depth"] - snap_df["no_depth"]) / (snap_df["yes_depth"] + snap_df["no_depth"] + 1)

# To test: pair snapshots with subsequent trade prices
# Load trade data to measure price movement after each snapshot
print("Loading live WS trades for order book imbalance analysis...")

# Load bronze trades
paginator = s3.get_paginator("list_objects_v2")
trade_keys = []
for page in paginator.paginate(Bucket=S3_BUCKET, Prefix="bronze/kalshi_ws/trade/"):
    trade_keys.extend([o["Key"] for o in page.get("Contents", [])])
    if len(trade_keys) >= 200:
        break

print(f"Loading {min(len(trade_keys), 100)} trade files...")
ws_trades = []
for key in trade_keys[:100]:
    obj = s3.get_object(Bucket=S3_BUCKET, Key=key)
    data = gzip.decompress(obj["Body"].read()).decode()
    for line in data.strip().split("\n"):
        rec = json.loads(line)
        msg = rec["frame"]["msg"]
        ws_trades.append({
            "t_receipt": rec["t_receipt"],
            "ticker": msg.get("market_ticker", ""),
            "yes_price": float(msg["yes_price_dollars"]) if "yes_price_dollars" in msg else None,
            "ts": msg.get("ts", ""),
        })

ws_trades_df = pd.DataFrame(ws_trades)
ws_trades_df["t_receipt_dt"] = pd.to_datetime(ws_trades_df["t_receipt"], unit="s", utc=True)
print(f"Loaded {len(ws_trades_df)} WS trades across {ws_trades_df['ticker'].nunique()} tickers")

In [ ]:
# For each ticker in snapshots, measure price movement in the 30s after the snapshot
# Join: for each snapshot, find trades in the next 30 seconds for the same ticker

# Only analyze tickers present in both snapshots and trades
common_tickers = set(snap_df["ticker"]) & set(ws_trades_df["ticker"])
print(f"Tickers in both snapshots and trades: {len(common_tickers)}")

# Debug: check time overlap between snapshots and trades
snap_time_range = (snap_df["t_receipt"].min(), snap_df["t_receipt"].max())
trade_time_range = (ws_trades_df["t_receipt"].min(), ws_trades_df["t_receipt"].max())
print(f"Snapshot time range: {pd.to_datetime(snap_time_range[0], unit='s', utc=True)} to {pd.to_datetime(snap_time_range[1], unit='s', utc=True)}")
print(f"Trade time range:    {pd.to_datetime(trade_time_range[0], unit='s', utc=True)} to {pd.to_datetime(trade_time_range[1], unit='s', utc=True)}")

# The snapshots are taken at subscription time, so they're sparse in time.
# Widen the window: look for trades within 5 MINUTES after each snapshot (not 30s)
# This measures medium-term price direction vs book state at subscription.

imbalance_results = []
for ticker in list(common_tickers):
    tk_snaps = snap_df[snap_df["ticker"] == ticker].sort_values("t_receipt")
    tk_trades = ws_trades_df[ws_trades_df["ticker"] == ticker].sort_values("t_receipt")
    
    if tk_trades.empty or tk_snaps.empty:
        continue
    
    for _, snap in tk_snaps.iterrows():
        t0 = snap["t_receipt"]
        # Find trades in [t0, t0+300s] (5 minutes)
        future_trades = tk_trades[(tk_trades["t_receipt"] > t0) & 
                                   (tk_trades["t_receipt"] <= t0 + 300)]
        if len(future_trades) < 2:
            continue
        
        # Price movement = last trade in window vs first trade
        price_first = future_trades["yes_price"].iloc[0]
        price_last = future_trades["yes_price"].iloc[-1]
        
        if price_first and price_last:
            # Use mid-price if available, else best_bid
            mid = (snap["best_bid"] + snap["best_ask"]) / 2 if snap["best_ask"] else snap["best_bid"]
            imbalance_results.append({
                "ticker": ticker,
                "imbalance": snap["imbalance"],
                "spread": snap["spread"],
                "price_move": price_last - price_first,
                "mid_to_last": price_last - mid if mid else None,
                "n_trades_5m": len(future_trades),
            })

imb_df = pd.DataFrame(imbalance_results)
print(f"\nImbalance → price movement (5-min window): {len(imb_df)} observations")

if len(imb_df) >= 10:
    # Bin by imbalance quintile
    n_bins = min(5, len(imb_df) // 3)
    imb_df["imb_bin"] = pd.qcut(imb_df["imbalance"], n_bins, labels=False, duplicates="drop")
    
    print(f"\nPrice movement by imbalance bin ({n_bins} bins):")
    print(f"{'Bin':<6} {'Mean Imb':<12} {'Mean Move':<12} {'N':<6}")
    print("-" * 40)
    for q, grp in imb_df.groupby("imb_bin"):
        print(f"{q:<6} {grp['imbalance'].mean():<12.3f} {grp['price_move'].mean():<12.4f} {len(grp):<6}")
    
    # Correlation
    corr = imb_df["imbalance"].corr(imb_df["price_move"])
    print(f"\nCorrelation(imbalance, 5m_price_move): {corr:.4f}")
    print(f"  (positive = yes-heavy book predicts price going up)")
    
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(imb_df["imbalance"], imb_df["price_move"], alpha=0.4, s=15)
    ax.axhline(0, color="gray", linestyle="--")
    ax.axvline(0, color="gray", linestyle="--")
    # Trend line
    z = np.polyfit(imb_df["imbalance"], imb_df["price_move"], 1)
    p = np.poly1d(z)
    x_range = np.linspace(imb_df["imbalance"].min(), imb_df["imbalance"].max(), 50)
    ax.plot(x_range, p(x_range), "r-", linewidth=2, label=f"trend (corr={corr:.3f})")
    ax.set_xlabel("Book Imbalance (yes_depth - no_depth) / total")
    ax.set_ylabel("Price Move (5-min window)")
    ax.set_title(f"Order Book Imbalance vs Future Price Movement")
    ax.legend()
    plt.tight_layout()
    plt.show()
elif len(imb_df) > 0:
    print(f"\nOnly {len(imb_df)} observations — not enough for quintile analysis.")
    print(f"This is a data limitation: snapshots are sparse (only at WS subscription).")
    print(f"Need continuous book state reconstruction from deltas for proper analysis.")
else:
    print("\nNo valid observations — snapshot and trade time windows don't overlap.")
    print("Snapshots happen at subscription time; trades may come later.")
    print("Need to reconstruct book from deltas for continuous imbalance signal.")

In [ ]:
# Market making profitability estimate
# If we can post at best_bid and best_ask and get filled on both sides,
# profit = spread - fee per round trip

# Estimate fill probability: how often does the price cross our posted level?
# Proxy: if a trade occurs at or below our posted bid (or at/above our ask) within N seconds

# For now, theoretical analysis:
valid_snap = snap_df[snap_df["spread"].notna() & (snap_df["spread"] > 0)].copy()
valid_snap["series"] = valid_snap["ticker"].apply(
    lambda t: re.match(r"(KXNBA\w+?)-", t).group(1) if re.match(r"(KXNBA\w+?)-", t) else "unknown"
)

print("MARKET MAKING FEASIBILITY")
print("=" * 60)
print(f"\nIf we capture the full spread and pay $0.02 fee per round-trip:")
print(f"  {'Series':<15} {'Med Spread':<12} {'Profit/RT':<12} {'% Profitable'}")
print(f"  {'-'*50}")

for series, grp in valid_snap.groupby("series"):
    med_spread = grp["spread"].median()
    profit = med_spread - 0.02
    pct_profitable = (grp["spread"] > 0.02).mean()
    print(f"  {series:<15} ${med_spread:.3f}       ${profit:.3f}       {pct_profitable:.0%}")

print(f"\n  NOTE: Assumes both legs fill (optimistic). Real fill rate depends on")
print(f"  adverse selection — you get filled when the market moves against you.")
print(f"  Net profit ≈ spread_captured × fill_rate - adverse_selection_cost")